## Load Data

In [1]:
# import libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# import data
DATA_DIR = Path.cwd().parents[0] / "Data"  # adjust if needed

def load_csv(filename: str) -> pd.DataFrame:
    fp = DATA_DIR / filename

    if fp.exists():
        print(f"Loaded: {fp}")
        return pd.read_csv(fp)
    raise FileNotFoundError(
        f"Could not find {filename} in any of: {[str(d) for d in DATA_DIR]}"
    )

df_movie = load_csv("MoviesOnStreamingPlatforms.csv")
df_tv = load_csv("TVShowsOnStreamingPlatforms.csv")
print("df_movie shape:", df_movie.shape)
print("df_tv shape:", df_tv.shape)

Loaded: /home/csantosh/Build_Projects/streaming-bi-template/Data/MoviesOnStreamingPlatforms.csv
Loaded: /home/csantosh/Build_Projects/streaming-bi-template/Data/TVShowsOnStreamingPlatforms.csv
df_movie shape: (9515, 15)
df_tv shape: (5368, 15)


In [3]:
df_movie.head()

,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type,Genre,Country,Language,IMDb,IMDb_ID
0,1,The Irishman,2019,18+,98/100,1,0,0,0,0,"Biography, Crime, Drama",United States,"English, Italian, Latin, Spanish, German",7.8,tt1302006
1,2,Dangal,2016,7+,97/100,1,0,0,0,0,"Action, Biography, Drama","India, United States","Hindi, English",8.3,tt5074352
2,3,David Attenborough: A Life on Our Planet,2020,7+,95/100,1,0,0,0,0,"Documentary, Biography",United Kingdom,English,8.9,tt11989890
3,4,Lagaan: Once Upon a Time in India,2001,7+,94/100,1,0,0,0,0,"Drama, Musical, Sport","India, United States","Hindi, English",8.1,tt0169102
4,5,Roma,2018,18+,94/100,1,0,0,0,0,Drama,"Mexico, United States","Spanish, Mixtec, English, Japanese, German, Fr...",7.6,tt6155172


In [4]:
df_movie.info()

<class 'pandas.DataFrame'>
RangeIndex: 9515 entries, 0 to 9514
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               9515 non-null   int64  
 1   Title            9515 non-null   str    
 2   Year             9515 non-null   int64  
 3   Age              5338 non-null   str    
 4   Rotten Tomatoes  9508 non-null   str    
 5   Netflix          9515 non-null   int64  
 6   Hulu             9515 non-null   int64  
 7   Prime Video      9515 non-null   int64  
 8   Disney+          9515 non-null   int64  
 9   Type             9515 non-null   int64  
 10  Genre            6030 non-null   str    
 11  Country          6387 non-null   str    
 12  Language         6026 non-null   str    
 13  IMDb             6732 non-null   float64
 14  IMDb_ID          6787 non-null   str    
dtypes: float64(1), int64(7), str(7)
memory usage: 1.1 MB


In [5]:
df_tv.info()

<class 'pandas.DataFrame'>
RangeIndex: 5368 entries, 0 to 5367
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   ID               5368 non-null   int64
 1   Title            5368 non-null   str  
 2   Year             5368 non-null   int64
 3   Age              3241 non-null   str  
 4   Rotten Tomatoes  5368 non-null   str  
 5   Netflix          5368 non-null   int64
 6   Hulu             5368 non-null   int64
 7   Prime Video      5368 non-null   int64
 8   Disney+          5368 non-null   int64
 9   Type             5368 non-null   int64
 10  Genre            2210 non-null   str  
 11  Country          2697 non-null   str  
 12  Language         2043 non-null   str  
 13  IMDb             4498 non-null   str  
 14  IMDb_ID          2688 non-null   str  
dtypes: int64(7), str(8)
memory usage: 629.2 KB


## Parsing and Conversion

In [6]:
# Common parsing / conversion helpers

def standardize_title(df: pd.DataFrame, col: str = "Title") -> pd.DataFrame:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip() # Removes leading and trailing whitespace
    return df 

def add_age_min(df: pd.DataFrame, source_col: str = "Age") -> pd.DataFrame:
    """Convert '18+' -> 18, '7+' -> 7 (numeric). Leaves NaN if missing/unparseable."""
    if source_col in df.columns:
        df["Age_Min"] = pd.to_numeric(
            df[source_col].astype(str).str.extract(r"(\d+)")[0],
            errors="coerce"
        )
    return df

def add_rotten_tomatoes_score(df: pd.DataFrame, source_col: str = "Rotten Tomatoes") -> pd.DataFrame:
    """Convert '98/100' -> 98.0 (numeric). Leaves NaN if missing/unparseable."""
    if source_col in df.columns:
        df["RottenTomatoes_Score"] = pd.to_numeric(
            df[source_col].astype(str).str.extract(r"(\d+)")[0], 
            errors="coerce"
        )
        
    return df

def standardize_type(df: pd.DataFrame, col: str = "Type") -> pd.DataFrame:
    """The datasets encode Type as 0/1. We'll map to readable labels."""
    if col in df.columns:
        # Keep original for reference
        df[col] = df[col].map({0: "movie", 1: "tv_show"}).fillna(df[col].astype(str))
    return df


In [7]:
# Apply conversions to BOTH datasets
for _df in (df_movie, df_tv):
    standardize_title(_df)
    add_age_min(_df)
    add_rotten_tomatoes_score(_df)
    standardize_type(_df)
    _df.drop(columns=['Age', 'Rotten Tomatoes'], inplace=True)
    
df_movie.head()

,ID,Title,Year,Netflix,Hulu,Prime Video,Disney+,Type,Genre,Country,Language,IMDb,IMDb_ID,Age_Min,RottenTomatoes_Score
0,1,The Irishman,2019,1,0,0,0,movie,"Biography, Crime, Drama",United States,"English, Italian, Latin, Spanish, German",7.8,tt1302006,18.0,98.0
1,2,Dangal,2016,1,0,0,0,movie,"Action, Biography, Drama","India, United States","Hindi, English",8.3,tt5074352,7.0,97.0
2,3,David Attenborough: A Life on Our Planet,2020,1,0,0,0,movie,"Documentary, Biography",United Kingdom,English,8.9,tt11989890,7.0,95.0
3,4,Lagaan: Once Upon a Time in India,2001,1,0,0,0,movie,"Drama, Musical, Sport","India, United States","Hindi, English",8.1,tt0169102,7.0,94.0
4,5,Roma,2018,1,0,0,0,movie,Drama,"Mexico, United States","Spanish, Mixtec, English, Japanese, German, Fr...",7.6,tt6155172,18.0,94.0


In [8]:
# tv shows data has IMDb in strings and needs to be converted to float
def imdb_score_to_float(df: pd.DataFrame, source_col: str = "IMDb") -> pd.DataFrame:
    """Convert '98/100' -> 98.0 (numeric). Leaves NaN if missing/unparseable."""
    if source_col in df.columns:
        df["IMDb_Score"] = pd.to_numeric(
            df[source_col].astype(str).str.extract(r"(\d+)")[0], 
            errors="coerce"
        )
    df.drop(columns=['IMDb'], inplace=True)
    df.rename(columns={'IMDb_Score':'IMDb'}, inplace=True)
        
    return df

In [9]:
df_tv = imdb_score_to_float(df_tv)

In [10]:
df_tv.head()

,ID,Title,Year,Netflix,Hulu,Prime Video,Disney+,Type,Genre,Country,Language,IMDb_ID,Age_Min,RottenTomatoes_Score,IMDb
0,1,Breaking Bad,2008,1,0,0,0,tv_show,"crime television series, drama television seri...",United States,"American English, Spanish language in the Amer...",tt0903747,18.0,100,10.0
1,2,Stranger Things,2016,1,0,0,0,tv_show,"science fiction television program, horror tel...",United States,English,tt4574334,16.0,96,9.0
2,3,Attack on Titan,2013,1,1,0,0,tv_show,NaN,NaN,NaN,NaN,18.0,95,9.0
3,4,Better Call Saul,2015,1,0,0,0,tv_show,"legal drama, crime television series",United States,English,tt3032476,18.0,94,10.0
4,5,Dark,2017,1,0,0,0,tv_show,"science fiction television program, drama tele...",Germany,German,tt5753856,16.0,93,10.0


## Removing duplicates based on the titles

In [11]:
# Here we use Title to remove duplicate entries with the same title
def dedupe(df: pd.DataFrame, subset_cols):
    before = len(df)
    df2 = df.drop_duplicates(subset=[c for c in subset_cols if c in df.columns], keep="first")
    after = len(df2)
    print(f"Removed {before - after} duplicates using subset={subset_cols}")
    return df2

df_movie = dedupe(df_movie, ["Title"])
df_tv = dedupe(df_tv, ["Title"])

Removed 0 duplicates using subset=['Title']
Removed 0 duplicates using subset=['Title']


## Handling Missing Data

In [12]:
# Handling only important columns with missing values
for _df in [df_movie, df_tv]:
    for col in ['IMDb', 'Age_Min', 'RottenTomatoes_Score']:
        _df[col] = _df[col].fillna(_df[col].median())

    for col in ['Genre', 'Country', 'Language']:
        mode = _df[col].mode(dropna=True)
        fill_val = mode.iloc[0] if len(mode) > 0 else "Unknown"
        _df[col] = _df[col].fillna(fill_val)
        
    

In [13]:
df_movie.info()

<class 'pandas.DataFrame'>
RangeIndex: 9515 entries, 0 to 9514
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    9515 non-null   int64  
 1   Title                 9515 non-null   str    
 2   Year                  9515 non-null   int64  
 3   Netflix               9515 non-null   int64  
 4   Hulu                  9515 non-null   int64  
 5   Prime Video           9515 non-null   int64  
 6   Disney+               9515 non-null   int64  
 7   Type                  9515 non-null   str    
 8   Genre                 9515 non-null   str    
 9   Country               9515 non-null   str    
 10  Language              9515 non-null   str    
 11  IMDb                  9515 non-null   float64
 12  IMDb_ID               6787 non-null   str    
 13  Age_Min               9515 non-null   float64
 14  RottenTomatoes_Score  9515 non-null   float64
dtypes: float64(3), int64(6), str(6)


In [14]:
df_tv.info()

<class 'pandas.DataFrame'>
RangeIndex: 5368 entries, 0 to 5367
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    5368 non-null   int64  
 1   Title                 5368 non-null   str    
 2   Year                  5368 non-null   int64  
 3   Netflix               5368 non-null   int64  
 4   Hulu                  5368 non-null   int64  
 5   Prime Video           5368 non-null   int64  
 6   Disney+               5368 non-null   int64  
 7   Type                  5368 non-null   str    
 8   Genre                 5368 non-null   str    
 9   Country               5368 non-null   str    
 10  Language              5368 non-null   str    
 11  IMDb_ID               2688 non-null   str    
 12  Age_Min               5368 non-null   float64
 13  RottenTomatoes_Score  5368 non-null   int64  
 14  IMDb                  5368 non-null   float64
dtypes: float64(2), int64(7), str(6)


In [15]:
df_movie.head()

,ID,Title,Year,Netflix,Hulu,Prime Video,Disney+,Type,Genre,Country,Language,IMDb,IMDb_ID,Age_Min,RottenTomatoes_Score
0,1,The Irishman,2019,1,0,0,0,movie,"Biography, Crime, Drama",United States,"English, Italian, Latin, Spanish, German",7.8,tt1302006,18.0,98.0
1,2,Dangal,2016,1,0,0,0,movie,"Action, Biography, Drama","India, United States","Hindi, English",8.3,tt5074352,7.0,97.0
2,3,David Attenborough: A Life on Our Planet,2020,1,0,0,0,movie,"Documentary, Biography",United Kingdom,English,8.9,tt11989890,7.0,95.0
3,4,Lagaan: Once Upon a Time in India,2001,1,0,0,0,movie,"Drama, Musical, Sport","India, United States","Hindi, English",8.1,tt0169102,7.0,94.0
4,5,Roma,2018,1,0,0,0,movie,Drama,"Mexico, United States","Spanish, Mixtec, English, Japanese, German, Fr...",7.6,tt6155172,18.0,94.0


In [16]:
df_tv.head()

,ID,Title,Year,Netflix,Hulu,Prime Video,Disney+,Type,Genre,Country,Language,IMDb_ID,Age_Min,RottenTomatoes_Score,IMDb
0,1,Breaking Bad,2008,1,0,0,0,tv_show,"crime television series, drama television seri...",United States,"American English, Spanish language in the Amer...",tt0903747,18.0,100,10.0
1,2,Stranger Things,2016,1,0,0,0,tv_show,"science fiction television program, horror tel...",United States,English,tt4574334,16.0,96,9.0
2,3,Attack on Titan,2013,1,1,0,0,tv_show,reality television,United States,English,NaN,18.0,95,9.0
3,4,Better Call Saul,2015,1,0,0,0,tv_show,"legal drama, crime television series",United States,English,tt3032476,18.0,94,10.0
4,5,Dark,2017,1,0,0,0,tv_show,"science fiction television program, drama tele...",Germany,German,tt5753856,16.0,93,10.0
